In [7]:
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import re
import ast
import json
from run_utils import run_llm_judge

import random

In [4]:
# gpt-5 classified hard and easy queries

In [9]:
reader = open('testv2_configs_snippets.json', 'r')
query_json = json.loads(reader.read())
queries = [samp['question'] for samp in query_json]

In [10]:
indices_hard = [8, 71, 64, 73, 75]
for idx in indices_hard:
    print(f"{queries[idx]}")

print("\n\n")
indices_easy = [14, 28, 76, 84, 54]
for idx in indices_easy:
    print(f"{queries[idx]}")

My goal is to have a large generative machine learning model be good at reasoning with spatial dimensions to help with building 3d film scenes. This includes positioning and orienting objects well in a 3d environment. This could include use a large language model, a large multimodal model, or a large video generation model. The current approach I am taking is to develop a dataset with correct input and output using a top down view of a scene and position and orientating objects correctly. However, it is a manual tedious task and would prefer more automated synthetic generation approach. The end goal could be either to fine tune any of the models specified for this type of task. Ideally I would use a movie scene dataset which I have. There has been a lot of recent findings of the benefits of reasoning with chain-of-thought (CoT). Use recent research to suggest a few automated approaches.
I need a literature review that examines the intersection of blockchain technology and social networ

In [5]:
# sqa eval read json
with open('task_sqa_solver_sqa_claude_4.json', 'r') as f:
    sqa_results = json.load(f)

In [27]:
scores = []
for i in indices_hard:
    scores.append(sqa_results['samples'][i]['scores']['aggregate_scorer']['value']['ingredient_recall'])
print("Hard queries ingredient recall:", np.asarray(scores).mean())

scores = []
for i in indices_easy:
    scores.append(sqa_results['samples'][i]['scores']['aggregate_scorer']['value']['ingredient_recall'])
print("Easy queries ingredient recall:", np.asarray(scores).mean())

Hard queries ingredient recall: 0.7911344537815127
Easy queries ingredient recall: 0.8945751633986928


In [21]:
# sqa eval read json
with open('dr_tulu.json', 'r') as f:
    drtulu_results = json.load(f)

In [28]:
scores = []
for i in indices_hard:
    scores.append(drtulu_results['samples'][i]['scores']['aggregate_scorer']['value']['ingredient_recall'])
print("Hard queries ingredient recall:", np.asarray(scores).mean())

scores = []
for i in indices_easy:
    scores.append(drtulu_results['samples'][i]['scores']['aggregate_scorer']['value']['ingredient_recall'])
print("Easy queries ingredient recall:", np.asarray(scores).mean())

Hard queries ingredient recall: 0.8453501400560226
Easy queries ingredient recall: 0.8372630718954248


In [29]:
# sqa eval read json
with open('task_sqa_solver_openscilm.json', 'r') as f:
    os_results = json.load(f)

In [33]:
scores = []
for i in indices_hard:
    scores.append(os_results['samples'][i]['scores']['aggregate_scorer']['value']['global_avg'])
print("Hard queries average score:", np.asarray(scores).mean())

scores = []
for i in indices_easy:
    scores.append(os_results['samples'][i]['scores']['aggregate_scorer']['value']['global_avg'])
print("Easy queries average score:", np.asarray(scores).mean())

Hard queries average score: 0.5144051431775216
Easy queries average score: 0.5147049101650696


In [25]:
# sqa eval read json
with open('o3-deep-research-2025-06-26.json', 'r') as f:
    os_results = json.load(f)

scores = []
for i in indices_hard:
    scores.append(os_results['samples'][i]['scores']['score_all']['value']['global_avg'])
print("Hard queries average score:", np.asarray(scores).mean().round(2))

scores = []
for i in indices_easy:
    scores.append(os_results['samples'][i]['scores']['score_all']['value']['global_avg'])
print("Easy queries average score:", np.asarray(scores).mean().round(2))

scores = []
for i in indices_hard:
    scores.append(os_results['samples'][i]['scores']['score_all']['value']['ingredient_recall'])
print("Hard queries ingredient recall:", np.asarray(scores).mean().round(2))

scores = []
for i in indices_easy:
    scores.append(os_results['samples'][i]['scores']['score_all']['value']['ingredient_recall'])
print("Easy queries ingredient recall:", np.asarray(scores).mean().round(2))

Hard queries average score: 0.83
Easy queries average score: 0.82
Hard queries ingredient recall: 0.89
Easy queries ingredient recall: 0.95


In [34]:
sqa_responses = []
with open('sqa.json', 'r') as f:
    sqa_data = json.load(f)

for samp in sqa_data:
    answer = ""
    for sec in samp['response']['sections']:
        answer += sec['title'] +  "\n" + sec['text'] + "\n\n"
    sqa_responses.append(answer)

tulu_responses = []
with open('tulu.jsonl', 'r') as f:
    tulu_data = [json.loads(line) for line in f]

# sort tulu_data by example_id
tulu_data = sorted(tulu_data, key=lambda x: x['example_id'])

for i, samp in enumerate(tulu_data):
    assert sqa_data[i]['question'] == samp['problem']
    tulu_responses.append(samp['final_response'])

gpt_5_responses = []
with open('gpt_5_search_outputs.jsonl', 'r') as f:
    gpt_5_data = [json.loads(line) for line in f]

# sort gpt_5_data by example_id
gpt_5_data = sorted(gpt_5_data, key=lambda x: x['id'])

for i, samp in enumerate(gpt_5_data):
    assert sqa_data[i]['question'] == samp['problem']
    gpt_5_responses.append(samp['answer'])

In [35]:
model_name = "gpt-4.1"
all_samples = []
for i in range(len(sqa_data)):
    query = sqa_data[i]['question']
    
    if random.random() < 0.5:
        model_a_response = gpt_5_responses[i]
        model_b_response = tulu_responses[i]
        preferred_model = "dummy"
        all_samples.append([model_a_response, model_b_response, ["gpt-5", "tulu"], preferred_model, query])
    else:
        model_a_response = tulu_responses[i]
        model_b_response = gpt_5_responses[i]
        preferred_model = "dummy"
        all_samples.append([model_a_response, model_b_response, ["tulu", "gpt-5"], preferred_model, query])

results = run_llm_judge(all_samples, model_name)
    
    

  2%|▏         | 2/100 [00:07<05:22,  3.29s/it]

Could not decode JSON from response: {
"reasoning": Response A excels in providing a structured, readable, and thoroughly referenced chronological and thematic overview of ray tracing primarily focused on NURBS surfaces. It starts with historical methods, then covers numerically robust techniques, GPU-accelerated strategies, optical applications, and touches on relevant adjacent topics such as trimmed NURBS and geodesic/creeping-ray tracing. The citations are clearly tied to claims and easily followed, and the organization leads the reader through method advancements and applications step-by-step. It concludes with a summary and points out gaps and areas for further research.

Response B offers a comprehensive review but is less clear and self-contained in its citations (they are referenced as internal cite IDs, which appears incomplete and makes it hard for a reader to track sources). The text is detailed and technical but more fragmented, lacking the narrative clarity and coherent th

  5%|▌         | 5/100 [00:07<01:23,  1.13it/s]

Could not decode JSON from response: {
"reasoning": Response B is more comprehensive and rigorously evidence-based, providing a broader range of challenges to SME digitalisation globally. It not only tabulates the challenges in a clear format but also includes a wider variety of issues such as interoperability, legal and regulatory uncertainty, data governance, supply chain integration, market access, and inequality—many of which Response A only addresses partially or omits. Response B also explicitly discusses evidence, providing multiple in-text citations for each listed challenge, and follows up with detailed implications and policy levers, which enhances both relevance and completeness. The organization is strong, with sections that move logically from challenges to implications, and the clarity is maintained throughout. Citations, while somewhat technical (with citation IDs rather than standard URLs), show thorough sourcing. Response A is very well-organized, clear, and gives a ro

  8%|▊         | 8/100 [00:08<00:40,  2.25it/s]

...

Could not decode JSON from response: {
"reasoning": Response A is notably superior to Response B on all main evaluation axes:

1. **Completeness**: Response A systematically covers all major variants of inspection games with one inspector and several inspectees, including static, dynamic, Stackelberg/commitment, imperfect detection, mean-field (large N), and patrolling/Stackelberg scheduling. It also addresses algorithmic solution techniques and gives step-by-step practitioner guidance for implementation. Response B, while accurate, surveys fewer technical models and leaves out mean-field approaches, many algorithmic details, and dynamic recursive/finite resource scheduling.

2. **Relevance**: Both are relevant, but A gives much more actionable insight, differentiates convex vs. concave detection models, covers imperfect inspection, and gives a detailed, practical framework for use.

3. **Clarity**: Both are readable and well-organized. A is denser and requires more technical backgroun

 98%|█████████▊| 98/100 [00:40<00:00,  2.45it/s]

Could not decode JSON from response: {
"reasoning": Response B is the stronger answer for the following reasons. Both answers are highly complete, detailed, and contain a broad overview of Flow Matching Models, with references to theoretical foundations, objective functions, relationship to diffusion models, and practical extensions. However, B stands out for several reasons:
1. **Completeness**: While both cover core ideas, B covers practical details such as guidance (classifier-free guidance, guidance mechanisms in discrete structure), explicit points on how FM enables both ODE and SDE-based sampling, and discusses conditioning, guidance strategies, and multiple coupling/path designs. It elaborates on practical limitations and open problems—giving a critical perspective.
2. **Relevance**: B consistently relates the technical explanation to practical considerations for both training and inference, and effectively presents the connections and differences between FM, diffusion, and CNF 

100%|██████████| 100/100 [00:46<00:00,  2.14it/s]

Could not decode JSON from response: {
"reasoning": 
Both responses tackle the core question: "How does a 1 bit LLM work?" Each provides ample technical detail about low-bit (notably 1-bit and ternary/1.58-bit) quantization, training and inference tradeoffs, implications for accuracy, and hardware/system acceleration. However, Response B does a significantly better job in several key areas:

- **Completeness**: Response B not only covers the basic principles of 1-bit quantization and model design but also provides much more technical depth into how weights and activations are quantized, what happens in training, how errors are compensated with feedback/error theories, hybrid schemes (e.g., selector/value vectors), and hardware co-design. It covers end-to-end pipeline from architecture through training and inference in granular detail and even explains why matmul can reduce to addition.
- **Relevance**: All explanations in B are highly relevant, clearly relating to how a 1-bit LLM works

In [36]:
np.unique(np.asarray([samp['chosen_response'] for samp in results]), return_counts=True)

(array(['gpt-5', 'tulu'], dtype='<U5'), array([47, 53]))

In [42]:
results[2]

{'query': 'Are there any articles focusing on the use of attention mechanisms in ranking? Where the weights generated by the attention mechanism are fused as weights for the base ranking?',
 'models': ['tulu', 'gpt-5'],
 'chosen_response': 'tulu',
 'model_response': '{\n"reasoning": "Both answers are well-researched and thorough, but Response A is superior in completeness, breadth, and specificity. Response A provides a much broader and more nuanced survey, citing a wide range of primary research articles across classic learning-to-rank (LTR, ANN, Attention Rank, DLCM), transformer-based listwise rankers (RankFormer), and recommender/ad systems (DIN, DIEN, DHAN, AFM, AFAFM). It connects these concretely to the mechanics of attention-derived weights (for fusing representations or features for final ranking prediction) and explains in detail how these mechanisms operate in each paradigm, giving practical guidance and tying citations to each key claim. The answer is organized with section

In [55]:
all_samples

[['# Overview: What “Diagnosis” Means for VLMs like CLIP\n\nDiagnosis of vision–language models (VLMs) such as CLIP spans two complementary goals: explaining and localizing what evidence drives predictions, and auditing failure modes such as spurious correlations, out-of-distribution (OOD) brittleness, and hallucinations that arise in model use or fine-tuning pipelines.<cite id="9fd0fb73-0">Understanding CLIP representations has moved from high-level alignment to component-level analyses that decompose latent embeddings into human-interpretable concepts, examine attention heads and MLP neurons, and use sparse autoencoders (SAEs) to expose diverse, representation-aligned components</cite>.<cite id="3c8ac24f-0">Recent robustness studies show that while CLIP appears robust to ImageNet-style shifts, specialized benchmarks reveal reliance on spurious features that can break under real-world distribution shifts</cite>.<cite id="3c8ac24f-4">Complementary lines of work investigate object hallu